In [ ]:
!pip install -U transformers datasets accelerate evaluate

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from transformers import Trainer, TrainingArguments
from datasets import load_dataset
import json
import wandb
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import math
import evaluate

# Initialising wandb
wandb.init(project="debertha-v3-base-fine-tuning", name="debertha-email-analyzer")

# Initiating the checkpoint
CHECKPOINT = "google-bert/bert-base-uncased"

# Initiating the model
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT,
                                  num_labels=3,
                                  id2label={0:"low", 1:"medium", 2:"high"},
                                  label2id={"low":0, "medium":1, "high":2})
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

# Initiating wandb
# wandb.init(project="debertha-v3-base-fine-tuning", name="debertha-email-analyzer")

def tokenize(batch):
  return tokenizer(
      batch["subject"],
      batch["body"],
      truncation="only_second",
      max_length=512
  )

def compute_metrics(eval_pred):
  logits = eval_pred.predictions
  labels = eval_pred.label_ids

  predictions = np.argmax(logits, axis=-1)

  return {
      "accuracy": accuracy_score(labels, predictions),
      "f1": f1_score(labels, predictions, average="weighted"),
  }


def main():
  # Load the dataset
  raw_dataset = load_dataset("json", data_files="drive/MyDrive/Datasets/preprocessed_emails_json_data.json")

  # Split the data into training data and validation data
  split_dataset = raw_dataset["train"].train_test_split(test_size=0.2, seed=42)
  train_dataset = split_dataset["train"]
  val_dataset = split_dataset["test"]

  # Tokenize the email subject and body for training data
  tokenized_train_dataset = train_dataset.map(tokenize,
                                      batched=True,
                                      remove_columns=[col for col in train_dataset.column_names if col != "urgency"])

  # Tokenize the email subject and body for validation data
  tokenized_val_dataset = val_dataset.map(tokenize,
                                      batched=True,
                                      remove_columns=[col for col in val_dataset.column_names if col != "urgency"])

  # Rename urgency column to labels
  tokenized_train_dataset = tokenized_train_dataset.rename_column("urgency", "labels")
  tokenized_val_dataset = tokenized_val_dataset.rename_column("urgency", "labels")

  # Initiate the training argument to contain all hyperparameters for our trainer
  training_args = TrainingArguments(
      output_dir="results",
      eval_strategy ="steps",
      warmup_steps = 500,
      learning_rate=2e-5,
      eval_steps=50,
      save_steps=100,
      logging_steps=10,  # Log metrics every 10 steps
      num_train_epochs=3,
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      load_best_model_at_end=True,
      report_to="wandb"
  )

  # Initiate the data collator
  data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

  # Initiate the trainer
  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset = tokenized_train_dataset,
      eval_dataset = tokenized_val_dataset,
      data_collator=data_collator,
      processing_class=tokenizer,
      compute_metrics=compute_metrics
  )

  # Train the model
  trainer.train()

  # Save the model and tokenizer to a folder
  trainer.save_model("trained_model_2")

  # Also save the tokenizer separately
  tokenizer.save_pretrained("trained_model_2")

if __name__ == "__main__":
  main()

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import numpy as np

model_path = "trained_model"
id2label = {0:"low", 1:"medium", 2:"high"}

model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

model.eval()

subject = "Legal documents"
body = "Hi Jane. Can you provide me with the legal documents for Pty Ltd please? Thanks."

text = subject + " " + body

input_ids = tokenizer(text, truncation=True, padding=True, max_length=512, return_tensors ="pt")

with torch.no_grad():
  outputs = model(**input_ids)
  logits = outputs.logits
  prediction = torch.argmax(logits, dim=-1).item()

  predicted_label = id2label[prediction]

print("Predicted urgency:", predicted_label)